# Bi-Mamba zh↔vi — Notebook xuất biểu đồ báo cáo

Notebook **độc lập** với pipeline train (không cần thay đổi gì trong `src/`).
Đầu vào duy nhất là:

* **Stdout của ô Train** trong notebook chính (`bi_mamba_zh_vi_colab.ipynb`).
  Trainer in ra mỗi `eval_every` step một dòng dạng
  `step N | val_loss V.VVV (best B.BBB) | ema_val_loss V.VVV (best B.BBB)`.
  Ta parse các dòng này → đường cong `val_loss` + `ema_val_loss`.
* **Thư mục `runs/bi_mamba_55m/`** chứa các checkpoint đã lưu (`best.pt`,
  `best_ema.pt`, `avg_last5.pt`, `avg_last5_ema.pt`, `latest.pt`,
  `latest_ema.pt`). Dùng để (a) đếm số tham số và (b) chạy
  `scripts/evaluate.py` lấy BLEU/chrF.

Notebook sẽ xuất:
1. **Đường cong loss** (`val_loss` + `ema_val_loss` theo step, đánh dấu best).
2. **LR schedule** — tái dựng giải tích từ config (cosine + warmup, không cần log).
3. **Bảng số tham số** — embedding / encoder / decoder / norms.
4. **BLEU + chrF** chạy `evaluate.py` cho 6 checkpoint + 1 ensemble decode 3
   checkpoint mạnh nhất.
5. **Vài câu dịch mẫu** (qualitative).

Tất cả figure → `reports/figures/*.png`, bảng → `reports/summary.csv`.


## 1. Mount Drive (tuỳ chọn) + clone repo

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print('Skipping drive mount:', e)


In [ ]:
import os
REPO_URL = 'https://github.com/ChauDucToan/NLP_DHM.git'
REPO_DIR = '/content/NLP_DHM'
if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull --ff-only || true
%cd $REPO_DIR
!pwd && ls


## 2. Cài đặt dependencies

In [ ]:
!pip install -q -r requirements.txt
!pip install -q matplotlib pandas
import sys; sys.path.insert(0, 'src')
import torch
print('torch:', torch.__version__,
      '| CUDA:', torch.cuda.is_available(),
      '| device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')


## 3. Cấu hình đường dẫn

`RUN_DIR` là nơi chứa checkpoint. `TRAIN_LOG` là file text bạn lưu stdout của
ô Train (xem mục 4 dưới). Nếu đã copy `runs/bi_mamba_55m/` lên Drive, đổi
đường dẫn cho phù hợp.

In [ ]:
from pathlib import Path
import yaml

CONFIG_PATH = Path('configs/_colab.yaml') if Path('configs/_colab.yaml').exists() else Path('configs/bi_mamba_55m.yaml')
RUN_DIR     = Path('runs/bi_mamba_55m')
TOKEN_DIR   = Path('data/tokenizer')
TRAIN_LOG   = Path('reports/training_stdout.log')   # bạn paste stdout vào file này (xem mục 4)
FIG_DIR     = Path('reports/figures'); FIG_DIR.mkdir(parents=True, exist_ok=True)
TRAIN_LOG.parent.mkdir(parents=True, exist_ok=True)

# Ưu tiên Drive nếu đã được copy lên đó.
DRIVE_RUN_DIR = Path('/content/drive/MyDrive/bi_mamba_55m')
if DRIVE_RUN_DIR.exists():
    print(f'Using Drive run dir: {DRIVE_RUN_DIR}')
    RUN_DIR = DRIVE_RUN_DIR
    if (DRIVE_RUN_DIR / 'tokenizer' / 'spm.model').exists():
        TOKEN_DIR = DRIVE_RUN_DIR / 'tokenizer'
    drive_log = DRIVE_RUN_DIR / 'training_stdout.log'
    if drive_log.exists():
        TRAIN_LOG = drive_log
        print(f'Using Drive log: {TRAIN_LOG}')

print(f'CONFIG    = {CONFIG_PATH}')
print(f'RUN_DIR   = {RUN_DIR}')
print(f'TOKEN_DIR = {TOKEN_DIR}')
print(f'TRAIN_LOG = {TRAIN_LOG}  (exists={TRAIN_LOG.exists()})')
print(f'FIG_DIR   = {FIG_DIR}')
print()
print('Files in run dir:')
for p in sorted(RUN_DIR.glob('*'))[:30]:
    if p.is_file():
        print(f'  {p.name:30s} {p.stat().st_size/1e6:.1f} MB')


## 4. Cung cấp stdout của ô Train

Có **2 cách** để có file `TRAIN_LOG`:

**Cách A — paste trực tiếp vào ô bên dưới** (nhanh, không cần Drive).
Mở notebook gốc → click chuột phải vào output của ô Train → *Copy output* →
dán vào ô bên dưới (giữa 2 dấu `'''`).

**Cách B — đọc từ file `.log`** (đã `wget`/`copy` lên Colab hoặc Drive).
Bỏ qua cell A, đảm bảo `TRAIN_LOG` ở cell trên trỏ đúng tới file.

In [ ]:
# === CÁCH A: PASTE STDOUT VÀO ĐÂY ===
# Để trống nếu bạn đã có file ở TRAIN_LOG (cách B).
PASTED_STDOUT = '''
# Ví dụ vài dòng (paste hết stdout của ô Train của bạn vào đây):
#
# step 2000 | val_loss 4.752 (best 4.752) | ema_val_loss 4.812 (best 4.812)
#   -> new best val_loss 4.7521 @ step 2000 (saved best.pt)
#   -> new best ema_val_loss 4.8123 @ step 2000 (saved best_ema.pt)
# step 4000 | val_loss 3.581 (best 3.581) | ema_val_loss 3.762 (best 3.762)
#   ...
'''

if PASTED_STDOUT.strip() and 'step' in PASTED_STDOUT and 'val_loss' in PASTED_STDOUT:
    TRAIN_LOG.write_text(PASTED_STDOUT)
    print(f'Wrote {len(PASTED_STDOUT):,} bytes to {TRAIN_LOG}')
else:
    print(f'Skipping — using existing file at {TRAIN_LOG} (exists={TRAIN_LOG.exists()})')


## 5. Parse log → DataFrame

Trainer in ra 2 dạng dòng:
* `step N | val_loss V.VVV (best B.BBB) | ema_val_loss V.VVV (best B.BBB)` — khi EMA bật.
* `step N | val_loss V.VVV (best B.BBB)` — khi EMA tắt.

Regex bên dưới bắt cả 2 dạng.

In [ ]:
import re
import pandas as pd

assert TRAIN_LOG.exists() and TRAIN_LOG.stat().st_size > 0, (
    f'Không tìm thấy log ở {TRAIN_LOG}. Quay lại mục 4 paste stdout, '
    f'hoặc trỏ TRAIN_LOG sang file đúng.'
)

PAT = re.compile(
    r'step\s+(\d+)\s+\|\s+val_loss\s+([\d.]+)\s+\(best\s+([\d.]+)\)'
    r'(?:\s+\|\s+ema_val_loss\s+([\d.]+)\s+\(best\s+([\d.]+)\))?'
)

rows = []
for line in TRAIN_LOG.read_text(errors='replace').splitlines():
    m = PAT.search(line)
    if not m:
        continue
    d = {
        'step': int(m.group(1)),
        'val_loss': float(m.group(2)),
        'best_val_loss': float(m.group(3)),
    }
    if m.group(4):
        d['ema_val_loss']      = float(m.group(4))
        d['best_ema_val_loss'] = float(m.group(5))
    rows.append(d)

df_eval = pd.DataFrame(rows).drop_duplicates(subset=['step']).sort_values('step').reset_index(drop=True)
print(f'Parsed {len(df_eval)} eval points from log.')
if len(df_eval):
    print(f'  Steps: {df_eval.step.min():,} → {df_eval.step.max():,}')
    print(f'  Best val_loss = {df_eval.best_val_loss.min():.4f} '
          f'@ step {int(df_eval.loc[df_eval.val_loss.idxmin(), "step"]):,}')
    if 'best_ema_val_loss' in df_eval:
        print(f'  Best ema_val_loss = {df_eval.best_ema_val_loss.min():.4f} '
              f'@ step {int(df_eval.loc[df_eval.ema_val_loss.idxmin(), "step"]):,}')
df_eval.tail()


## 6. Loss curves (val + EMA-val)

In [ ]:
import matplotlib.pyplot as plt

assert len(df_eval) > 0, 'No data — paste log first.'

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(df_eval.step, df_eval.val_loss, marker='o', ms=4, color='C0', label='val_loss')
if 'ema_val_loss' in df_eval:
    ax.plot(df_eval.step, df_eval.ema_val_loss, marker='s', ms=4, color='C1', label='ema_val_loss')

# Đánh dấu best.
bs = int(df_eval.loc[df_eval.val_loss.idxmin(), 'step'])
bv = float(df_eval.val_loss.min())
ax.axvline(bs, color='C0', ls='--', lw=0.7, alpha=0.5)
ax.annotate(f'best val_loss={bv:.3f}\n@ step {bs:,}',
            xy=(bs, bv), xytext=(8, 14), textcoords='offset points',
            fontsize=9, color='C0')
if 'ema_val_loss' in df_eval:
    bs2 = int(df_eval.loc[df_eval.ema_val_loss.idxmin(), 'step'])
    bv2 = float(df_eval.ema_val_loss.min())
    ax.axvline(bs2, color='C1', ls='--', lw=0.7, alpha=0.5)
    ax.annotate(f'best ema={bv2:.3f}\n@ step {bs2:,}',
                xy=(bs2, bv2), xytext=(8, -22), textcoords='offset points',
                fontsize=9, color='C1')

ax.set_xlabel('Training step')
ax.set_ylabel('Cross-entropy loss')
ax.set_title('Bi-Mamba zh↔vi — validation loss')
ax.grid(True, alpha=0.3)
ax.legend(loc='upper right', frameon=True)
fig.tight_layout()
out = FIG_DIR / 'loss_curves.png'
fig.savefig(out, dpi=150, bbox_inches='tight')
print(f'Saved: {out}')
plt.show()


## 7. LR schedule (tái dựng giải tích)

Dùng cùng công thức `cosine_lr` của trainer + config thực tế. Không cần parse log.

In [ ]:
import math
import yaml
import numpy as np

cfg = yaml.safe_load(open(CONFIG_PATH))
tr  = cfg['train']

def cosine_lr(step, warmup, max_steps, base_lr, min_lr):
    if step < warmup:
        return base_lr * (step + 1) / max(1, warmup)
    p = (step - warmup) / max(1, max_steps - warmup)
    p = min(max(p, 0.0), 1.0)
    return min_lr + 0.5 * (base_lr - min_lr) * (1.0 + math.cos(math.pi * p))

# Chọn endpoint = step cuối quan sát được trong log (nếu có), nếu không = max_steps.
endpoint = int(df_eval.step.max()) if len(df_eval) else int(tr['max_steps'])
xs = np.linspace(0, endpoint, num=400, dtype=int)
ys = [cosine_lr(int(s), tr['warmup_steps'], tr['max_steps'], tr['lr'], tr['min_lr']) for s in xs]

fig, ax = plt.subplots(figsize=(11, 3.5))
ax.plot(xs, ys, color='C2', lw=1.4)
ax.axvspan(0, tr['warmup_steps'], color='C2', alpha=0.07, label=f"warmup ({tr['warmup_steps']:,})")
ax.set_yscale('log'); ax.set_xlabel('Training step'); ax.set_ylabel('Learning rate (log)')
ax.set_title(f"LR schedule — warmup→cosine  (base={tr['lr']:.0e}, min={tr['min_lr']:.0e}, max_steps={tr['max_steps']:,})")
ax.grid(True, which='both', alpha=0.3); ax.legend()
fig.tight_layout()
out = FIG_DIR / 'lr_schedule.png'
fig.savefig(out, dpi=150, bbox_inches='tight')
print(f'Saved: {out}')
plt.show()


## 8. Bảng số tham số của model

Phân loại theo embedding (chia sẻ với lm_head), encoder (Bi-Mamba × N),
decoder (Mamba + cross-attention × N), norms.

In [ ]:
import torch
from bi_mamba_mt.model import BiMambaTranslator, ModelConfig

ckpt_for_arch = None
for cand in ['best_ema.pt', 'avg_last5_ema.pt', 'best.pt', 'latest.pt']:
    if (RUN_DIR / cand).exists():
        ckpt_for_arch = RUN_DIR / cand; break
assert ckpt_for_arch is not None, f'Không tìm thấy bất kỳ .pt nào trong {RUN_DIR}'

ckpt = torch.load(ckpt_for_arch, map_location='cpu', weights_only=False)
mcfg = ModelConfig(**ckpt['model_cfg'])
model = BiMambaTranslator(mcfg)

def _sum(prefix):
    return sum(p.numel() for n, p in model.named_parameters() if n.startswith(prefix))

stats = {
    'embedding (tied with lm_head)':    _sum('embedding.'),
    'encoder (Bi-Mamba × N)':            _sum('encoder_layers.'),
    'decoder (Mamba + cross-attn × N)':  _sum('decoder_layers.'),
    'final norms':                       _sum('encoder_norm.') + _sum('decoder_norm.'),
}
total = sum(stats.values())

import pandas as pd
breakdown = pd.DataFrame(
    [(k, v, 100*v/total) for k, v in stats.items()] + [('TOTAL', total, 100.0)],
    columns=['component', 'params', '% of total']
)
breakdown['params'] = breakdown['params'].map(lambda x: f'{x:,}')
print(f'Architecture: d_model={mcfg.d_model}, encoder={mcfg.n_encoder_layers}, '
      f'decoder={mcfg.n_decoder_layers}, d_ff={mcfg.d_ff}, vocab={mcfg.vocab_size}')
breakdown.to_csv('reports/param_breakdown.csv', index=False)
breakdown


## 9. BLEU + chrF cho từng checkpoint (+ ensemble decode)

Chạy `scripts/evaluate.py` cho mỗi checkpoint + 1 lần ensemble 3 checkpoint
mạnh nhất, parse stdout → bar chart. Default 1000 cặp test, mất ~5–15 phút
trên T4 / RTX 6000.

In [ ]:
import os, re, json, subprocess

RESULTS_PATH = Path('reports/eval_results.json')
RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)

EVAL_N = 1000  # giảm xuống nếu muốn nhanh hơn

CKPTS = [
    ('latest',         RUN_DIR / 'latest.pt'),
    ('latest_ema',     RUN_DIR / 'latest_ema.pt'),
    ('best',           RUN_DIR / 'best.pt'),
    ('best_ema',       RUN_DIR / 'best_ema.pt'),
    ('avg_last5',      RUN_DIR / 'avg_last5.pt'),
    ('avg_last5_ema',  RUN_DIR / 'avg_last5_ema.pt'),
]
CKPTS = [(n, p) for n, p in CKPTS if p.exists()]

# Ensemble: 3 checkpoint tốt nhất (nếu đủ hỗ trợ trong evaluate.py).
ENSEMBLE_NAMES = ['best_ema', 'avg_last5_ema', 'best']
ensemble_paths = [p for n, p in CKPTS if n in ENSEMBLE_NAMES]
do_ensemble = len(ensemble_paths) >= 2

# Detect xem evaluate.py có hỗ trợ --checkpoints không (PR #5 thêm) — nếu
# master chưa có, fallback chỉ chạy single-checkpoint.
help_out = subprocess.run(['python', 'scripts/evaluate.py', '--help'],
                          capture_output=True, text=True).stdout
SUPPORTS_ENSEMBLE = '--checkpoints' in help_out
if not SUPPORTS_ENSEMBLE and do_ensemble:
    print('NOTE: scripts/evaluate.py chưa hỗ trợ --checkpoints (chỉ có ở PR Path A). '
          'Bỏ qua phần ensemble cho an toàn.')
    do_ensemble = False

pat = re.compile(r'\[(\w+)\]\s+n=(\d+)\s+BLEU=([\d.]+)\s+chrF=([\d.]+)\s+\(beam=(\d+),\s*lp=([\d.]+)\)')

def _run(label, args):
    print(f'\n=== {label} ===')
    cmd = ['python', 'scripts/evaluate.py', '--config', str(CONFIG_PATH),
           '--num-samples', str(EVAL_N)] + args
    print('  $', ' '.join(cmd))
    out = subprocess.run(cmd, capture_output=True, text=True)
    print(out.stdout[-1500:])
    if out.returncode != 0:
        print('STDERR:', out.stderr[-1000:])
    parsed = {}
    for m in pat.finditer(out.stdout):
        d, n, bleu, chrf, beam, lp = m.groups()
        parsed[d] = {'n': int(n), 'bleu': float(bleu), 'chrf': float(chrf),
                     'beam': int(beam), 'lp': float(lp)}
    return parsed

results = {}
for name, ckpt in CKPTS:
    results[name] = _run(name, ['--checkpoint', str(ckpt)])

if do_ensemble:
    label = f'ensemble x{len(ensemble_paths)}'
    results[label] = _run(label, ['--checkpoints'] + [str(p) for p in ensemble_paths])

RESULTS_PATH.write_text(json.dumps(results, indent=2, ensure_ascii=False))
print(f'\nSaved: {RESULTS_PATH}')
results


## 10. Bar chart: BLEU + chrF theo checkpoint (cả 2 chiều)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

results = json.loads(RESULTS_PATH.read_text())
order = [k for k in [
    'latest', 'latest_ema',
    'best', 'best_ema',
    'avg_last5', 'avg_last5_ema',
] if k in results] + [k for k in results if k.startswith('ensemble')]

zh2vi_bleu = [results[k].get('zh2vi', {}).get('bleu', None) for k in order]
vi2zh_bleu = [results[k].get('vi2zh', {}).get('bleu', None) for k in order]
zh2vi_chrf = [results[k].get('zh2vi', {}).get('chrf', None) for k in order]
vi2zh_chrf = [results[k].get('vi2zh', {}).get('chrf', None) for k in order]

x = np.arange(len(order)); w = 0.38
fig, axes = plt.subplots(1, 2, figsize=(14, 4.6))

ax = axes[0]
ax.bar(x - w/2, zh2vi_bleu, w, label='zh→vi', color='C0')
ax.bar(x + w/2, vi2zh_bleu, w, label='vi→zh', color='C3')
for i, v in enumerate(zh2vi_bleu):
    if v is not None: ax.text(i - w/2, v + 0.4, f'{v:.1f}', ha='center', fontsize=8)
for i, v in enumerate(vi2zh_bleu):
    if v is not None: ax.text(i + w/2, v + 0.4, f'{v:.1f}', ha='center', fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(order, rotation=20, ha='right')
ax.set_ylabel('BLEU'); ax.set_title('SacreBLEU (n=1000) — both directions')
ax.grid(True, axis='y', alpha=0.3); ax.legend()

ax = axes[1]
ax.bar(x - w/2, zh2vi_chrf, w, label='zh→vi', color='C0')
ax.bar(x + w/2, vi2zh_chrf, w, label='vi→zh', color='C3')
for i, v in enumerate(zh2vi_chrf):
    if v is not None: ax.text(i - w/2, v + 0.4, f'{v:.1f}', ha='center', fontsize=8)
for i, v in enumerate(vi2zh_chrf):
    if v is not None: ax.text(i + w/2, v + 0.4, f'{v:.1f}', ha='center', fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(order, rotation=20, ha='right')
ax.set_ylabel('chrF'); ax.set_title('chrF (n=1000) — both directions')
ax.grid(True, axis='y', alpha=0.3); ax.legend()

fig.tight_layout()
out = FIG_DIR / 'bleu_chrf_per_ckpt.png'
fig.savefig(out, dpi=150, bbox_inches='tight')
print(f'Saved: {out}')
plt.show()


## 11. Bảng tóm tắt (CSV cho báo cáo)

In [ ]:
rows = []
for name in order:
    r = results.get(name, {})
    rows.append({
        'checkpoint':       name,
        'zh→vi BLEU':       r.get('zh2vi', {}).get('bleu'),
        'zh→vi chrF':       r.get('zh2vi', {}).get('chrf'),
        'vi→zh BLEU':       r.get('vi2zh', {}).get('bleu'),
        'vi→zh chrF':       r.get('vi2zh', {}).get('chrf'),
    })
summary = pd.DataFrame(rows)
summary.to_csv('reports/summary.csv', index=False)
summary


## 12. Vài câu dịch mẫu (qualitative)

In [ ]:
import torch
from bi_mamba_mt.model import BiMambaTranslator, ModelConfig
from bi_mamba_mt.tokenizer import Tokenizer
from bi_mamba_mt.translate import translate_batch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
demo_pri = ['avg_last5_ema.pt', 'best_ema.pt', 'best.pt', 'latest.pt']
demo_path = next((RUN_DIR / p for p in demo_pri if (RUN_DIR / p).exists()), None)
print(f'Demo checkpoint: {demo_path}')
ckpt = torch.load(demo_path, map_location='cpu', weights_only=False)
m = BiMambaTranslator(ModelConfig(**ckpt['model_cfg'])).to(device)
m.load_state_dict(ckpt['model'], strict=False); m.eval()
tok = Tokenizer(str(TOKEN_DIR / 'spm.model'))

demos_zh2vi = [
    '我们今天去吃越南河粉。',
    '人工智能正在改变世界。',
    '她每天早上六点起床去跑步。',
    '机器翻译还有很多需要改进的地方。',
]
demos_vi2zh = [
    'Hôm nay trời rất đẹp.',
    'Trí tuệ nhân tạo đang thay đổi thế giới.',
    'Cô ấy đã đi ngủ từ rất sớm.',
    'Học máy giúp giải quyết nhiều vấn đề khó.',
]

print('\n--- zh → vi ---')
out_zh2vi = translate_batch(m, tok, demos_zh2vi, 'zh2vi', beam_size=6, length_penalty=1.20, device=device)
for s, t in zip(demos_zh2vi, out_zh2vi):
    print(f'  {s}\n    -> {t}')

print('\n--- vi → zh ---')
out_vi2zh = translate_batch(m, tok, demos_vi2zh, 'vi2zh', beam_size=6, length_penalty=0.80, device=device)
for s, t in zip(demos_vi2zh, out_vi2zh):
    print(f'  {s}\n    -> {t}')

samples = {
    'zh2vi': [{'src': s, 'hyp': t} for s, t in zip(demos_zh2vi, out_zh2vi)],
    'vi2zh': [{'src': s, 'hyp': t} for s, t in zip(demos_vi2zh, out_vi2zh)],
    'checkpoint': str(demo_path.name),
}
Path('reports/qualitative_samples.json').write_text(json.dumps(samples, indent=2, ensure_ascii=False))
print('\nSaved: reports/qualitative_samples.json')


## 13. Done — files cho báo cáo

```
reports/
├── figures/
│   ├── loss_curves.png
│   ├── lr_schedule.png
│   └── bleu_chrf_per_ckpt.png
├── eval_results.json
├── param_breakdown.csv
├── summary.csv
└── qualitative_samples.json
```
